In [ ]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [ ]:
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
models = ["openrouter/auto",  "gpt-oss:20b-cloud", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b"]
clients = {"openrouter/auto": openrouter, "gpt-oss:20b-cloud": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter, "openai/gpt-oss-120b": ollama}

In [ ]:
language_map = {
    "Rust": {
        "extension": "rs",
        "compile_command": "rustc main.rs -O -o main && ./main",
    },
    "C++": {
        "extension": "cpp",
        "compile_command": "g++ main.cpp -O3 -std=c++17 -o main && ./main",
    },
}

language_options = ["Rust", "C++"]
default_language = "Rust"

In [ ]:
def get_system_prompt(language):
    return f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""


def user_prompt_for(python, language):
    language_meta = language_map[language]
    extension = language_meta["extension"]
    compile_command = language_meta["compile_command"]
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
Your response will be written to a file called main.{extension} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""


In [ ]:
def comment_system_prompt():
    return"""
You are an expert developer.
Your task is to add clear, concise, and useful comments to the given code.

Guidelines:
- Do NOT change the logic
- Only add comments
- Explain complex parts
- Keep comments minimal but meaningful
- Return only the updated code
"""

def comment_user_prompt(code, language):
    return f"""Here is some {language} code that needs comments:
{code}
"""

In [ ]:
def testCase_generator_system_prompt():
    return """You are an expert developer.
Your task is to generate test cases for the given code. You have to wxecute and tell user if the test cases passed or failed.
Guidelines:
- Do NOT change the logic
- Only add test cases
"""

def testCase_generator_user_prompt(code, language):
    return f"""Here is some {language} code that needs test cases:
{code}
"""

In [ ]:
def explain_system_prompt():
    return """
You are a highly skilled software engineer and coding instructor.

Your role is to explain code clearly, accurately, and in a structured way so that:
- Beginners can understand it
- Intermediate developers can learn optimization techniques

Guidelines:
- Be precise and technically correct
- Use simple language wherever possible
- Break down complex logic step-by-step
- Focus on understanding, not just description
- Highlight important patterns and concepts
- Include time and space complexity analysis
- Suggest better approaches if applicable

Do NOT:
- Modify or rewrite the original code
- Add unnecessary fluff or long paragraphs
- Use overly academic or confusing language

Always structure your response in a clean, readable format.
"""

def explain_prompt(code, language):
    return f"""
You are an expert software engineer and teacher.

Explain the following {language} code in a clear, structured, and beginner-friendly way.

Follow this format:

1. 🧾 Overview
- What does this code do?

2. ⚙️ Step-by-step Explanation
- Explain the logic in simple terms
- Break down important parts

3. 🧠 Key Concepts
- Mention important concepts used (loops, recursion, data structures, etc.)

4. ⏱ Time & Space Complexity
- Time Complexity: O(...)
- Space Complexity: O(...)
- Brief explanation why

5. 🚀 Possible Optimizations
- Suggest if the code can be improved
- Mention better approaches if any

6. 💡 Example (if helpful)
- Show a small example input/output

Rules:
- Keep it simple and easy to understand
- Avoid unnecessary jargon
- Be concise but clear
- Do NOT rewrite the full code

Code:
{code}
"""

In [ ]:
def explain_code(model, code, language):
    client = clients[model]
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": explain_system_prompt()},
            {"role": "user", "content": explain_prompt(code, language)}
        ]
    )
    return response.choices[0].message.content

In [ ]:
def add_comments(model, code, language):
    client = clients[model]
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": comment_system_prompt()},
            {"role": "user", "content": comment_user_prompt(code, language)}
        ]
    )
    return response.choices[0].message.content

In [ ]:
def add_test_cases(model, code, language):
    client = clients[model]
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": testCase_generator_system_prompt()},
            {"role": "user", "content": testCase_generator_user_prompt(code, language)}
        ]
    )
    return response.choices[0].message.content

In [ ]:

def messages_for(python, language):
    return [
        {"role": "system", "content": get_system_prompt(language)},
        {"role": "user", "content": user_prompt_for(python, language)}
    ]
 

In [ ]:
def port(model, python, language):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python, language), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```c++','').replace('```rust','').replace('```','')
    return reply

In [ ]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

# AI Code Converter Analyzer

This notebook provides a UI for converting Python code to Rust or C++, adding comments, generating test cases, and explaining the generated code.

## How to use
- Paste Python code into the left panel
- Choose the target language
- Click Convert to generate output code
- Optionally use the comment, test case, or explain buttons


In [ ]:
from styles import CSS

with gr.Blocks(
    css=CSS,
    theme=gr.themes.Monochrome(),
    title="AI Code Optimizer"
) as ui:

    # 🔥 Header
    gr.Markdown("""
    # 🚀 AI Code Optimizer + Explainer
    Paste your code → Optimize ⚡ → Understand 🧠 → Convert 🔄
    """)

    # 🧾 Code Section
    with gr.Row(equal_height=True):

        # LEFT: Input
        with gr.Column(scale=6):
            gr.Markdown("### Input Code")
            python = gr.Code(
                label="📝 Input Code (Python)",
                value=python_hard,
                language="python",
                lines=25
            )

        # RIGHT: Output + Explanation (stacked properly)
        with gr.Column(scale=6):
            gr.Markdown("### Output Code & Explanation")
            generated_code = gr.Code(
                label="⚡ Output Code",
                value="",
                lines=16,
                language="cpp"
            )

            explained_code = gr.Markdown(
                label="🧠 Explanation",
                value=""
            )

    # 🎛 Controls
    with gr.Row(elem_classes=["controls"]):
        gr.Markdown("### Controls")

        model = gr.Dropdown(
            models,
            value=models[0],
            label="🤖 Model",
            scale=2
        )

        language_selector = gr.Dropdown(
            label="🎯 Target",
            choices=language_options,
            value=default_language,
            scale=2
        )

        convert = gr.Button("🔄 Convert", elem_classes=["primary-btn"])
        comment = gr.Button("💬 Add Comments")
        testCases = gr.Button("🧪 Add Test Cases")
        explain_coder = gr.Button("🧠 Explain Code")

    # 🔗 Actions
    convert.click(
        fn=port,
        inputs=[model, python, language_selector],
        outputs=[generated_code]
    )

    comment.click(
        fn=add_comments,
        inputs=[model, generated_code, language_selector],
        outputs=[generated_code]
    )

    testCases.click(
        fn=add_test_cases,
        inputs=[model, generated_code, language_selector],
        outputs=[generated_code]
    )

    explain_coder.click(
        fn=explain_code,
        inputs=[model, generated_code, language_selector],
        outputs=[explained_code]
    )

ui.launch(inbrowser=True)
